In [1]:
!pip install -q google-generativeai

In [2]:
from google.colab import userdata
import google.generativeai as genai

# Load API key from Colab Secrets
api_key = userdata.get("GEMINI_API")
genai.configure(api_key=api_key)

# System prompt — gives the AI its personality and rules
system_prompt = """
You are a helpful, friendly Python tutor named PyBot.
You explain code clearly with examples.
Always be concise and encouraging.
"""

# Safety settings — list of dicts format required in SDK 0.8.x
safety_settings = [
    {"category": "HARM_CATEGORY_HARASSMENT",       "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH",       "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

# Create model with system prompt
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    system_instruction=system_prompt,
)

# Start chat session — this remembers conversation history
chat = model.start_chat(history=[])

print("✅ Ready! SDK version:", genai.__version__)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Ready! SDK version: 0.8.6


In [3]:
# ──────────────────────────────────────────
# TOOL DEFINITION
# ──────────────────────────────────────────
def get_weather(city: str) -> str:
    data = {
        "Mumbai": "32°C, Humid",
        "Delhi": "28°C, Sunny",
        "Aurangabad": "30°C, Clear skies",
        "London": "14°C, Rainy",
    }
    return data.get(city.strip().title(), f"No weather data found for {city}.")

weather_tool = {
    "function_declarations": [{
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]
        }
    }]
}

# ✅ Updated to gemini-2.0-flash
tool_model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    system_instruction="You are a helpful assistant. Use get_weather when asked about weather.",
    tools=[weather_tool],
)

# ──────────────────────────────────────────
# STREAMING FUNCTION
# ──────────────────────────────────────────
def stream_reply(user_message):
    print("🤖 PyBot: ", end="", flush=True)
    full_reply = ""

    response = chat.send_message(
        user_message,
        stream=True,
        safety_settings=safety_settings
    )

    for chunk in response:
        try:
            if hasattr(chunk, "text") and chunk.text:
                print(chunk.text, end="", flush=True)
                full_reply += chunk.text
        except Exception:
            continue

    print()
    if not full_reply:
        print("⚠️ Empty response.")

# ──────────────────────────────────────────
# TOOL USE FUNCTION
# ──────────────────────────────────────────
def tool_reply(user_message):
    response = tool_model.generate_content(user_message)
    part = response.candidates[0].content.parts[0]

    if hasattr(part, "function_call") and part.function_call:
        func_name = part.function_call.name
        func_args = dict(part.function_call.args)
        print(f"🔧 Tool called: {func_name}({func_args})")

        result = get_weather(**func_args) if func_name == "get_weather" else "Unknown tool"
        print(f"📦 Tool result: {result}")

        final = tool_model.generate_content([
            {"role": "user",  "parts": [{"text": user_message}]},
            {"role": "model", "parts": [part]},
            {"role": "user",  "parts": [{"function_response": {"name": func_name, "response": {"result": result}}}]}
        ])
        print(f"🤖 Assistant: {final.text}")
    else:
        print(f"🤖 Assistant: {response.text}")

# ──────────────────────────────────────────
# MAIN CHAT LOOP
# ──────────────────────────────────────────
print("💬 ChatBot started! Type 'quit' to exit.\n")

while True:
    user_input = input("🧑 You: ").strip()

    if not user_input:
        continue

    if user_input.lower() == "quit":
        print("👋 Bye!")
        break

    elif any(word in user_input.lower() for word in ["weather", "temperature", "forecast"]):
        tool_reply(user_input)

    else:
        stream_reply(user_input)

💬 ChatBot started! Type 'quit' to exit.

🧑 You: weather in mumbai
🔧 Tool called: get_weather({'city': 'mumbai'})
📦 Tool result: 32°C, Humid
🤖 Assistant: The weather in Mumbai is 32°C and humid.
🧑 You: what is a decorator in python
🤖 PyBot: That's a great question!

A **decorator** in Python is a special type of function that allows you to **wrap** another function.

Think of it like putting a fancy frame around a picture: the frame (decorator) adds extra features or modifies how the picture (your original function) is presented, without actually changing the picture itself.

**Why use them?**

They're super useful for adding extra functionality to existing functions, like:

*   Logging when a function is called
*   Measuring execution time
*   Adding authentication/authorization checks
*   Memoization (caching results)

**How it looks (the `@` syntax):**

```python
# 1. This is our decorator function
def my_decorator(func):
    def wrapper():
        print("Something is happening befor